# 교안 01-4: SQLite MCP 서버로 DB 조회하기

## 핵심 목표

DB 를 MCP 로 붙여, **스키마부터 물어 가며** 에이전트가 스스로 SELECT 를 쓰게 만든다.

## 학습 순서

1. 실습용 샘플 DB(chinook) 준비
2. SQLite MCP 서버 연결과 도구 6개: **읽기 도구와 쓰기 도구가 함께 온다**
3. 스키마부터 보기: 표 목록과 표 하나의 열 구성
4. SELECT 를 직접 보내 결과 받기
5. **읽기 도구만** 붙인 에이전트가 스스로 SQL 쓰기

## 쓰는 MCP 서버와 공식 문서

| 서버 | 실행 | 전송 | 공식 문서 |
|---|---|---|---|
| SQLite `mcp-server-sqlite` | `uvx` | stdio | https://pypi.org/project/mcp-server-sqlite/ |

## 준비물

- **uv**(`uvx --version`). 없으면 https://docs.astral.sh/uv/
- 인터넷. 첫 실행은 **DB 내려받기와 서버 설치**로 수십 초 걸립니다.
- **에이전트를 만드는 절부터 `OPENAI_API_KEY`** 가 필요합니다(일차 폴더의 `.env`).

---
## 1. 샘플 DB 준비

**chinook** 은 가상의 음악 판매점 데이터로, 표 11개가 서로 이어져 있는 SQLite 학습용 샘플 DB 입니다.
`data` 폴더에 함께 들어 있고, `utils.py` 의 `chinook_db_path()` 가 그 경로를 돌려줍니다.

DB 파일이 없으면 SQLite 서버는 **빈 DB 를 열고** 에러 문자열을 돌려줍니다.
예외가 아니라 평범한 문자열이라 모델이 그것을 결과로 착각합니다. 그래서 파일 준비를 맨 앞에서 끝냅니다.

In [ ]:
import sys
from pathlib import Path

# 노트북에는 __file__ 이 없다. 주피터는 노트북이 있는 폴더를 작업 폴더로 잡아 주므로 그 위가 일차 폴더다.
DAY_DIR = Path.cwd().parent        # 일차 폴더(day21). 아래 경로들의 기준점
sys.path.append(str(DAY_DIR))   # 일차 폴더의 utils.py 를 쓴다

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI

from utils import block_text, chinook_db_path, load_api_key, print_trajectory, quiet_stdio_logs

quiet_stdio_logs()          # 서버가 stdout 에 섞어 보내는 안내문 때문에 나는 긴 경고를 끈다
load_api_key(DAY_DIR)       # 모델을 부르는 절이 있으므로 키를 맨 앞에서 확인한다

DATA_DIR = DAY_DIR / "data"             # 실습에 쓰는 CSV·DB 가 있는 곳
DB_PATH = chinook_db_path(DATA_DIR)   # data 폴더의 chinook.db 경로를 돌려준다

---
## 서버 설정: 인자가 낯선 이유

이 설정은 앞의 서버들보다 인자가 깁니다. 한 줄씩 뜯어보면 두 덩어리입니다.
**어떤 패키지를 어떻게 띄울지**(`--with`·`--from`)와 **서버 자신의 인자**(`--db-path`)입니다.

| 키 | 값 | 뜻 |
|---|---|---|
| `command` | `"uvx"` | 파이썬 패키지를 받아 실행하는 실행기 |
| `args[0:2]` | `"--with", "mcp==1.9.4"` | 이 서버를 띄울 때 **mcp 라이브러리 버전을 고정**한다. 최신으로 띄우면 서버가 시작하자마자 죽는다 |
| `args[2:4]` | `"--from", "mcp-server-sqlite"` | **이 패키지에서** 실행 파일을 찾으라는 뜻 |
| `args[4]` | `"mcp-server-sqlite"` | 그 패키지 안에서 **실행할 명령 이름**. 패키지 이름과 명령 이름이 다를 수 있어 둘 다 적는다 |
| `args[5:7]` | `"--db-path", str(DB_PATH)` | **열어 줄 DB 파일 하나**. 서버가 정한 인자다 |
| `transport` | `"stdio"` | 자식 프로세스로 띄우고 표준입출력으로 대화 |

버전을 고정하는 이유가 이 실습의 숨은 교훈입니다. 이 서버는 **관리가 멈춰 있어서** 최신 `mcp` 라이브러리로 띄우면
서버가 쓰는 함수(`Server.list_resources`)가 없어져 그 자리에서 죽습니다.
남이 만든 서버를 쓸 때 실제로 겪는 일입니다. **버전을 고정할 자리를 알아 두는 것**이 대처법입니다.

In [ ]:
# SQLite 서버: DB 파일 하나를 열어 SQL 로 조회하게 해 준다.
# 01 번에서 파일시스템 서버에 폴더 하나만 준 것과 같은 방식이다. 지정한 파일 밖은 건드리지 못한다.
SQLITE = {
    "command": "uvx",                        # 파이썬 패키지를 받아 실행하는 실행기(uv 에 딸려 온다)
    "args": ["--with", "mcp==1.9.4",         # mcp 버전 고정. 최신으로 띄우면 서버가 시작하자마자 죽는다
             "--from", "mcp-server-sqlite",  # 이 패키지에서
             "mcp-server-sqlite",            # 이 명령을 실행한다(패키지 이름과 명령 이름이 다를 수 있다)
             "--db-path", str(DB_PATH)],     # 열어 줄 DB 파일 하나. 서버가 정한 인자다
    "transport": "stdio",                    # 내 컴퓨터에 프로세스로 띄운다
}

print("열어 줄 DB:", DB_PATH.name)

---
## 2. 서버에 붙어 도구 목록 받기

이 서버는 **상태가 없습니다**. 조회할 때마다 새로 붙어도 결과가 같으므로 `get_tools()` 로 받습니다.

In [ ]:
print("서버를 띄우는 중입니다(첫 실행은 오래 걸립니다)...")
tools = await MultiServerMCPClient({"db": SQLITE}).get_tools()
by_name = {tool.name: tool for tool in tools}

print(f"도구 {len(tools)}개")
for tool in tools:
    print(f" - {tool.name}({', '.join(tool.args)}): {tool.description.strip().splitlines()[0][:60]}")

도구를 성격별로 나눠 보면 이렇습니다.

| 갈래 | 도구 | 하는 일 |
|---|---|---|
| **읽기** | `list_tables` | 표 이름 목록 |
| | `describe_table` | 표 하나의 열 구성 |
| | `read_query` | `SELECT` 실행 |
| **쓰기** | `write_query` | `INSERT`·`UPDATE`·`DELETE` 실행 |
| | `create_table` | 표 만들기 |
| | `append_insight` | 서버가 들고 있는 메모에 한 줄 덧붙이기 |

**서버 하나에 읽기와 쓰기가 함께 들어 있습니다.** 01 번처럼 "이 폴더까지만" 하고 인자로 좁힐 방법이 없습니다.
이 목록을 통째로 에이전트에 넘기면 모델이 `write_query` 로 데이터를 지울 수도 있습니다.
그래서 뒤에서 **읽기 도구 세 개만** 골라 넘깁니다.

실무에서는 한 걸음 더 나갑니다. **DB 계정 자체를 읽기 전용으로** 만들어 둡니다.
"도구를 안 준다"는 우리 쪽 약속이지만, 계정 권한은 DB 가 강제하는 사실이기 때문입니다.

---
## 3. 스키마부터 보기

우리가 만든 데이터가 아니므로 **무슨 표가 있고 어떤 열을 가졌는지부터** 물어봐야 합니다.
사람이 그렇듯 모델도 마찬가지입니다. 뒤에서 모델에게 "먼저 스키마를 확인하라"고 시키는 근거가 여기 있습니다.

In [ ]:
# list_tables: 인자 없이 불러 DB 안의 표 이름을 모두 받아 오는 도구.
# 인자가 없는 도구도 호출 방식은 같다 - 빈 딕셔너리를 넘긴다.
# sqlite_ 로 시작하는 내부 표까지 함께 오므로, 걸러 보는 것은 받는 쪽 몫이다.
print(block_text(await by_name["list_tables"].ainvoke({})))

In [ ]:
# describe_table: 표 이름 하나를 받아 그 표의 열 구성을 돌려주는 도구.
# 인자 이름은 table_name 이다(table 이 아니다). 앞에서 찍어 본 인자 이름을 그대로 쓴다.
# 여기서 얻은 열 이름(BillingCountry·Total)이 있어야 앞의 질의를 쓸 수 있다.
print(block_text(await by_name["describe_table"].ainvoke({"table_name": "invoices"})))

---
## 4. SELECT 를 직접 보내기

`read_query` 는 **`SELECT` 만** 받습니다. 인자는 `query` 하나입니다.
`invoices` 한 표에 국가(`BillingCountry`)와 금액(`Total`)이 다 있어 조인이 필요 없습니다.
`ROUND` 로 소수 둘째 자리에서 반올림해 **대조하기 좋은 값**으로 고정합니다.

In [ ]:
country_sales_sql = """
SELECT BillingCountry AS country,
       COUNT(*) AS invoice_count,
       ROUND(SUM(Total), 2) AS total_sales
FROM invoices
GROUP BY BillingCountry
ORDER BY total_sales DESC
LIMIT 5
"""

print(block_text(await by_name["read_query"].ainvoke({"query": country_sales_sql})))

**대조표**: USA 91건 523.06 / Canada 56건 303.96 / France 35건 195.1 / Brazil 35건 190.1 / Germany 28건 156.48

값이 다르면 DB 파일이 배포본의 그 chinook 이 맞는지 먼저 의심하세요.

### 🖐️ 함께 따라하기: 스키마를 먼저 보고 질의 쓰기

데모는 `invoices` 표 하나만 봤습니다. 이번엔 **표 두 개를 이어야** 하는 질문을 손으로 풀어 봅니다.

1. `describe_table` 을 `table_name="albums"` 로 불러 어떤 열이 있는지 출력하세요.
2. 같은 도구로 `artists` 의 열도 확인하세요. **두 표를 잇는 열 이름**이 무엇인지 찾아 보세요.
3. 그 열로 두 표를 이어, **아티스트별 앨범 수 상위 5개**를 구하는 SQL 을 만들어 `read_query` 로 실행하세요.
   동점이 있으니 `ORDER BY` 에 아티스트 이름을 하나 더 더해 순서를 고정하세요.

**확인 기준**: Iron Maiden 21장, Led Zeppelin 14장이 위에 옵니다.
열 이름을 외워서 쓰지 말고 **1~2번에서 찍어 본 출력**을 보고 쓰세요. 그게 이 절의 요령입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) describe_table 을 table_name="albums" 로 불러 열 구성을 출력한다
# 2) 같은 도구로 artists 의 열도 확인하고, 두 표를 잇는 열을 찾는다
# 3) 그 열로 조인해 아티스트별 앨범 수 상위 5개를 구하는 SQL 을 read_query 로 실행한다

---
## 5. 에이전트가 스스로 SQL 을 쓰게 하기

앞에서 말한 대로 **읽기 도구만** 골라 넘깁니다. 넘기지 않은 도구는 모델이 **존재조차 모릅니다**.

시스템 프롬프트에 **"먼저 스키마를 확인하라"** 를 순서로 못 박습니다.
이 문장이 없으면 모델은 열 이름을 짐작해 질의를 쓰고, `no such column` 을 받은 뒤에야 뒤늦게 스키마를 물어보느라
호출을 낭비합니다.

In [ ]:
read_only_names = {"list_tables", "describe_table", "read_query"}
read_tools = [tool for tool in tools if tool.name in read_only_names]
print("에이전트에 붙일 도구:", [tool.name for tool in read_tools])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent(
    model,
    read_tools,
    system_prompt=(
        "너는 SQLite DB 분석 도우미다. 질문을 받으면 먼저 list_tables 와 describe_table 로 "
        "스키마를 확인한 뒤 SELECT 문을 작성해 read_query 로 실행한다. "
        "열 이름을 짐작하지 않고, 조회 결과에 없는 값을 지어내지 않는다. "
        "마지막 답은 조회한 표의 값을 근거로 이름과 수치를 함께 넣은 한국어 문장으로 쓴다."
    ),
)
print("에이전트 준비 완료")

In [ ]:
# 고객마다 담당 직원이 정해져 있다는 사실만 알려 주고, 어느 열인지는 모델이 스키마에서 찾게 둔다.
# '중복 없이'를 적어 두는 이유는 아래 해설에 있다. 이 한 마디가 없으면 고객 수가 부풀어 나온다.
question = (
    "고객마다 담당 직원이 지정돼 있어. 담당 직원별로 맡은 고객 수와 그 고객들의 매출 합계를 "
    "구해서 직원 이름과 함께 알려 줘. 고객 수는 같은 고객을 여러 번 세지 말고 중복 없이 세어 줘. "
    "담당 고객이 없는 직원은 빼고 알려 줘."
)
print("질문:", question, "\n")

# 메시지 기록에 모델이 만든 SQL 이 그대로 남는다. 사람이 검토할 수 있다는 점이 중요하다.
result = await agent.ainvoke({"messages": question})
print_trajectory(result)

**보는 법**: 도구 호출 줄의 `query` 인자가 모델이 쓴 SQL 입니다.
직원 이름은 `employees`, 담당 관계는 `customers` 의 `SupportRepId`, 매출은 `invoices` 에 있으므로 **표 세 개**를 이어야 합니다.
우리가 열 이름을 알려 주지 않았는데도 모델이 `SupportRepId` 를 찾아냈다면, 그것은 **스키마를 먼저 본 덕분**입니다.

**함정**: 질문에 "중복 없이"를 적은 이유가 여기 있습니다. `invoices` 를 이어 붙이는 순간 고객 한 명이
**주문 건수만큼 여러 줄로 늘어나서**, 그냥 세면 담당 고객 21명이 146명으로 부풀어 나옵니다.
`COUNT(DISTINCT CustomerId)` 로 세야 사람 수가 됩니다. 매출 합계는 줄마다 금액이 다르므로 그대로 `SUM` 하는 것이 맞습니다.
**같은 조인에서 열마다 셈법이 달라진다**는 점을 봐 두세요.

**대조표**: 담당 고객이 있는 직원은 셋뿐입니다. Jane Peacock 21명 833.04 / Margaret Park 20명 775.4 / Steve Johnson 18명 720.16

### 🖐️ 함께 따라하기: 계산을 어디서 했는지 확인하기

데모는 세는 일이었습니다. 이번엔 **계산이 들어간 질문**을 던져, 그 계산을 **DB 가 했는지 모델이 했는지** 확인합니다.

1. 위 `agent` 에게 물어보세요: **"가장 긴 트랙 3곡을 곡 이름과 함께 분 단위로 알려 줘."**
2. `print_trajectory()` 로 기록을 찍으세요.
3. 도구 호출 줄의 `query` 를 읽고, **나눗셈이 SQL 안에 있는지** 확인하세요.

**확인 기준**: SQL 안에 `Milliseconds / 60000.0` 같은 나눗셈이 있으면 **DB 가 나눈 값**을 그대로 옮긴 것입니다.
`Milliseconds` 만 조회해 왔다면 **모델이 밀리초를 받아 스스로 나눈 것**입니다.

답이 맞더라도 뒤쪽은 DB 가 아니라 **모델의 암산**을 믿는 셈이라, 계산이 길어질수록 틀리기 쉽습니다.
계산은 되도록 SQL 쪽에 시키는 편이 안전합니다. `60000` 처럼 정수로 나누면 소수점이 통째로 잘리는 것도 함께 확인해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) agent 에게 가장 긴 트랙 3곡을 분 단위로 물어본다
# 2) print_trajectory 로 기록을 찍는다
# 3) 도구 호출 줄의 query 를 읽고 나눗셈이 SQL 안에 있는지 확인한다

---
## 이번 실습 정리

| 배운 것 | 요점 |
|---|---|
| 서버 인자 | `--with`(라이브러리 버전 고정)·`--from`(패키지)·`--db-path`(서버 자신의 인자)가 섞여 있다 |
| 버전 고정 | 관리가 멈춘 서버는 최신 라이브러리에서 죽는다. 고정할 자리를 알아 둔다 |
| 스키마 먼저 | 우리가 만든 DB 가 아니다. `list_tables` → `describe_table` → `read_query` 순서 |
| 권한 좁히기 | 이 서버는 폴더처럼 좁힐 인자가 없다. **에이전트에 넘길 도구를 고르는 것**이 그 자리를 대신한다 |
| 조인의 함정 | 표를 이으면 줄이 늘어난다. 사람 수는 `COUNT(DISTINCT ...)`, 금액은 그대로 `SUM` |
| 검토 가능성 | 모델이 쓴 SQL 이 기록에 남아 사람이 읽고 검토할 수 있다 |

다음 실습: `05_코드실행_MCP.ipynb` 에서 **모델이 코드를 짜서** 계산하게 합니다.
DB 서버가 SQL 한 문장으로 끝내던 일을, 그쪽에서는 파이썬 코드를 보내 처리합니다. **통로를 고르는 감각**을 함께 보세요.